# 02 – Préparation : cible RUL, découpage par moteur, normalisation

Objectif : construire des jeux de données propres et **sans fuite d'information** pour les modèles.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.load import load_cmapss, add_rul, add_rul_test, PROJECT_ROOT
from src.data.split import split_units, apply_split
from src.features.preprocess import SensorScaler, FEATURES_FD001, DROP_COLS_FD001

sns.set_theme(style="whitegrid")
RUL_CLIP = 125

## 1. Cible RUL plafonnée

Les capteurs sont plats en début de vie puis la dégradation s'accélère (notebook 01). On plafonne donc la RUL à 125 : le modèle n'a pas à distinguer RUL = 300 et RUL = 200, ce qui est impossible avec des capteurs encore stables.

In [ ]:
train, test, rul_test = load_cmapss("FD001")
train = add_rul(train, clip=RUL_CLIP)
test = add_rul_test(test, rul_test, clip=RUL_CLIP)

raw = add_rul(train[train.unit == 1], clip=None)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(raw.cycle, raw.RUL, "--", label="RUL brute")
ax.plot(raw.cycle, train[train.unit == 1].RUL, lw=2, label=f"RUL plafonnée à {RUL_CLIP}")
ax.set(xlabel="Cycle", ylabel="RUL", title="Moteur 1 : cible avant / après plafonnement")
ax.legend()
plt.show()

## 2. Découpage par moteur

Règle d'or : un moteur appartient à **un seul** ensemble. Sinon le modèle verrait des cycles voisins du même moteur à l'entraînement et à l'évaluation, et les scores seraient trop optimistes.

In [ ]:
groups = split_units(train, fracs=(0.70, 0.15, 0.15), seed=42)
for k, u in groups.items():
    print(f"{k:<6}: {len(u):>3} moteurs -> {[int(x) for x in u[:8]]}...")

sets = apply_split(train, groups)
sets["test"] = test

# Vérification : aucun moteur commun
assert not set(groups["train"]) & set(groups["calib"])
assert not set(groups["train"]) & set(groups["val"])
assert not set(groups["calib"]) & set(groups["val"])
print("\nAucun moteur partagé entre les ensembles ✔")

In [ ]:
lifetimes = {k: v.groupby("unit").cycle.max() for k, v in sets.items() if k != "test"}
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(list(lifetimes.values()))
ax.set_xticks(range(1, len(lifetimes) + 1), list(lifetimes.keys()))
ax.set(ylabel="Durée de vie (cycles)", title="Les trois ensembles ont-ils des moteurs comparables ?")
plt.show()

## 3. Sélection des capteurs et normalisation

- Supprimés : op1, op2, op3, s1, s5, s6, s10, s16, s18, s19
- Gardés : 14 capteurs
- Normalisation ajustée **sur le train uniquement**, puis appliquée aux autres ensembles.

In [ ]:
scaler = SensorScaler(FEATURES_FD001).fit(sets["train"])
processed = {k: scaler.transform(v)[["unit", "cycle"] + FEATURES_FD001 + ["RUL"]] for k, v in sets.items()}

print("Moyennes après normalisation (train ≈ 0, autres proches de 0) :")
pd.DataFrame({k: v[FEATURES_FD001].mean() for k, v in processed.items()}).round(2)

## 4. Sauvegarde

In [ ]:
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
for k, v in processed.items():
    v.to_csv(out_dir / f"FD001_{k}.csv", index=False)
    print(f"{k:<6}: {v.shape} -> FD001_{k}.csv")